## Import Package

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from keras.utils import to_categorical
from keras.datasets import mnist
from keras.models import Sequential
from keras.layers import Dense
from ipywidgets import interact_manual

## Import Datasets

In [ ]:
(x_train_image ,y_train_label) ,(x_test_image ,y_test_label) = mnist.load_data()

## Confirmed Data Collected

In [ ]:
print("x_train_image:" ,x_train_image.shape)
print("y_train_label:" ,y_train_label.shape)

x_train_image: (60000, 28, 28)
y_train_label: (60000,)


In [ ]:
print("x_test_image:" ,x_test_image.shape)
print("y_test_label:" ,y_test_label.shape)

x_test_image: (10000, 28, 28)
y_test_label: (10000,)


In [ ]:
x_Train = x_train_image.reshape(60000 ,784).astype('float32')
x_Test = x_test_image.reshape(10000 ,784).astype('float32')

In [ ]:
print("x_Train:" ,x_Train.shape)
print("x_Test:" ,x_Test.shape)

x_Train: (60000, 784)
x_Test: (10000, 784)


## Normalize

In [ ]:
x_Train_normalize = x_Train / 255
x_Test_normalize = x_Test / 255

## One Hot Encoding

In [ ]:
y_Train_OneHot = to_categorical(y_train_label)
y_Test_OneHot = to_categorical(y_test_label)

## Model

Param

      = input_dim * units + Bias
      = 28*28 * 256 + 256
      = 200960

      = input_dim * units + Bias
      = 256 * 10 + 10
      = 2570

全連階層 unit == bias

In [ ]:
model = Sequential()

In [ ]:
model.add(Dense(units = 256,
                input_dim = 784,
                kernel_initializer = "normal",
                activation = "relu"))

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [ ]:
model.add(Dense(units = 10,
                kernel_initializer = "normal",
                activation = "softmax"))

In [ ]:
print(model.summary())

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_2 (Dense)                 │ (None, 256)            │       200,960 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 10)             │         2,570 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 203,530 (795.04 KB)

 Trainable params: 203,530 (795.04 KB)

 Non-trainable params: 0 (0.00 B)

None


In [ ]:
model.compile(loss = "categorical_crossentropy" ,
              optimizer = "adam",
              metrics = ["accuracy"])

In [ ]:
train_history = model.fit(x = x_Train_normalize,
                          y = y_Train_OneHot,
                          validation_split = 0.2,
                          epochs = 10,
                          batch_size = 200,
                          verbose = 0)

In [ ]:
scores = model.evaluate(x_Test_normalize ,y_Test_OneHot ,batch_size = 1)
print("Accrucy =", scores[1])

10000/10000 ━━━━━━━━━━━━━━━━━━━━ 18s 2ms/step - accuracy: 0.9790 - loss: 0.0699
Accrucy = 0.9789999723434448


In [ ]:
def statistics(x_normalize_data ,y_label_data ,batch_size=1) -> tuple:
  pre = model.predict(x_normalize_data ,batch_size)
  prediction = np.argmax(pre ,axis=1)
  conf_matrix = np.zeros((10, 10), dtype=int)

  for true_idx, pred_idx in zip(y_label_data, prediction):
    conf_matrix[true_idx, pred_idx] += 1

  return prediction ,conf_matrix

In [ ]:
prediction ,conf_matrix = statistics(x_Test_normalize ,y_test_label)

10000/10000 ━━━━━━━━━━━━━━━━━━━━ 13s 1ms/step


In [ ]:
errors_idx = []

for i in range(len(y_test_label)):
    if prediction[i] != y_test_label[i]:
        errors_idx.append(i)

In [ ]:
print("Conf_Matrix:")
print(conf_matrix)

Conf_Matrix:
[[ 968    0    1    1    1    0    3    1    2    3]
 [   0 1124    4    0    0    1    2    0    4    0]
 [   3    2 1009    3    1    0    2    9    3    0]
 [   0    0    1  987    0    6    0    5    3    8]
 [   2    0    5    1  952    1    1    4    2   14]
 [   3    0    0    4    1  873    5    1    3    2]
 [   6    3    1    1    3    6  937    0    1    0]
 [   0    3    7    1    0    0    0 1010    0    7]
 [   2    0    2    5    3    4    2    3  950    3]
 [   3    4    0    5    6    3    0    7    1  980]]


In [ ]:
print("Max combin: ",end="")

max_error = 0
pair = (0, 0)

for i in range(10):     # 真實數字
    for j in range(10): # 預測數字
        if i != j:      # 排除掉對角線 (判斷正確的情況)
            if conf_matrix[i, j] > max_error:
                max_error = conf_matrix[i, j]
                pair = (i, j)

print(f"將 {pair[0]} 誤判為 {pair[1]}，共計 {max_error} 次。")

Max combin: 將 4 誤判為 9，共計 14 次。


In [ ]:
def plot_image_labels_prediction(images=x_test_image,
                                 labels=y_test_label,
                                 prediction=prediction,
                                 idx=0,
                                 num=25) -> None:
  fig = plt.gcf()
  fig.set_size_inches(12 ,14)
  if num > 25: num = 25
  for i in range(0 ,num):
    ax = plt.subplot(5 ,5 ,1+i)
    ax.imshow(images[idx] ,cmap="binary")
    title = "label=" + str(labels[idx])
    title += ",predict=" + str(prediction[idx])
    ax.set_title(title ,fontsize=10)
    ax.set_xticks([])
    ax.set_yticks([])
    idx += 1
  plt.show

In [ ]:
def show_specific_errors(true_val, pred_val, count_num) -> None:
    # 1. 先找完：篩選出所有符合條件的索引
    specific_indices = []

    # 走訪每一筆資料的索引
    for i in range(len(y_test_label)):

        # 檢查：正確答案是不是我們要的？ 且 預測結果是不是我們要的？
        is_correct_target = (y_test_label[i] == true_val)
        is_predicted_target = (prediction[i] == pred_val)

        # 如果兩者都符合，就記錄這個位置
        if is_correct_target and is_predicted_target:
            specific_indices.append(i)

    total_found = len(specific_indices)
    print(f"查詢結果：真值={true_val}, 預測={pred_val} | 總計：{total_found} 筆")

    if total_found == 0:
        return

    # 2. 限制顯示數量最多為 25 (符合 5x5 格子)
    display_num = min(total_found, count_num, 25)

    # 3. 關鍵步驟：重新包裝資料
    # 我們只取出那些「誤判」的資料，組成一個新的暫時陣列
    temp_images = x_test_image[specific_indices[:display_num]]
    temp_labels = y_test_label[specific_indices[:display_num]]
    temp_preds = prediction[specific_indices[:display_num]]

    # 4. 呼叫原函式：
    # 因為 temp_images 的第 0 張就是我們要的第一個誤判
    # 所以傳入 idx=0, num=display_num
    # 這樣原函式內部的 subplot(5, 5, 1+i) 就會完美地水平排開！
    plot_image_labels_prediction(images=temp_images,
                                 labels=temp_labels,
                                 prediction=temp_preds,
                                 idx=0,
                                 num=display_num)

In [ ]:
# 互動介面
interact_manual(show_specific_errors,
                true_val=(0, 9, 1),
                pred_val=(0, 9, 1),
                count_num=(1, 25, 1));

interactive(children=(IntSlider(value=4, description='true_val', max=9), IntSlider(value=4, description='pred_…